# 데이터셋 검수 — 키보드로 에피소드 고르고 최종 데이터셋 만들기

`real_vr_teleop_record.ipynb` 로 모은 [LeRobot v3.0 데이터셋](https://huggingface.co/docs/lerobot/en/lerobot-dataset-v3)을 열어
에피소드를 하나씩 눈으로 확인하고, **보관/폐기**를 키보드로 정한 뒤 보관한 것만 담은 최종 데이터셋을 만듭니다.

원본은 **수정하지 않습니다** — `delete_episodes()` 가 새 디렉터리에 새 데이터셋을 씁니다.

| 키 | 동작 |
|---|---|
| `SPACE` | 재생 / 일시정지 |
| `K` | **보관**(keep) 하고 다음 에피소드 |
| `D` | **폐기**(discard) 하고 다음 에피소드 |
| `R` | 이 에피소드 처음부터 다시 |
| `.` 또는 `→` | 한 프레임 앞으로 (일시정지 중) |
| `N` / `P` | 판정 없이 다음 / 이전 에피소드 |
| `Q` | 검수 종료 (지금까지 판정은 유지) |

판정하지 않고 넘긴 에피소드는 **보관**으로 처리됩니다.

> 실행 순서: ① 준비 → ② 열기·요약 → ③ 키보드 검수 → ④ 판정 확인 → ⑤ 최종 생성 → ⑥ 검증

In [ ]:
# ① 준비 — 워크스페이스 찾기 (로봇 통신 없음)
from pathlib import Path
import runpy

_v2_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "VR_teleoperation")]
_v2_root = next((candidate for candidate in _v2_candidates
                 if (candidate / ".project_v2").is_file()), None)
if _v2_root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_v2_bootstrap = runpy.run_path(str(_v2_root / "package" / "init_project.py"))
for _v2_name in _v2_bootstrap["__all__"]:
    globals()[_v2_name] = _v2_bootstrap[_v2_name]

import numpy as np
import pandas as pd
import cv2
from ri_motion_v5_package.qt import use_pyqt_platform_plugins
use_pyqt_platform_plugins()   # Linux: cv2 창을 띄우기 전에 Qt 플러그인 경로를 맞춘다

## ② 데이터셋 열기 + 요약

`action` 은 텔레옵 명령입니다. 양손 grip 을 잡기 전 구간에는 명령이 없어 `NaN` 으로 기록되므로,
`action_NaN` 비율이 높은 에피소드는 실제 조작이 거의 없었다는 뜻입니다 — 폐기 후보로 먼저 보세요.

In [ ]:
# ② 데이터셋 열기 + 에피소드별 요약
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from package.episode_recorder import dataset_status, list_episodes, rejected_episodes

SAVE_ROOT = Path.home() / "aiworker_data"
TASK_NAME = "pick_and_place"
REPO_ID   = "local/%s" % TASK_NAME
root      = SAVE_ROOT / TASK_NAME

status = dataset_status(root)
print("경로:", root, "| 상태:", status)
if status != "ready":
    raise SystemExit(
        "검수할 데이터셋이 없습니다 (상태: %s). 'ready' 여야 합니다 — 녹화 노트북 종료 시 "
        "'finalized · N episodes' 메시지까지 기다렸는지 확인하세요." % status)

# video_backend="pyav": 이 PC 의 torchcodec 빌드는 ffmpeg 공유 라이브러리를 못 읽는다
ds = LeRobotDataset(REPO_ID, root=root, video_backend="pyav")
print("fps %d | 에피소드 %d | 프레임 %d | 카메라 %s"
      % (ds.fps, ds.meta.total_episodes, len(ds), list(ds.meta.video_keys)))

# action 의 NaN 비율은 parquet 에서 직접 읽는다 (비디오 디코딩 없이 빠르게)
table = pd.concat([pd.read_parquet(p) for p in sorted((root / "data").rglob("*.parquet"))],
                  ignore_index=True)
rows = []
for ep in list_episodes(root):
    index = ep["episode_index"]
    sub = table[table["episode_index"] == index]
    action = np.stack(sub["action"].to_numpy()) if len(sub) else np.zeros((0, 1))
    rows.append({"episode": index, "frames": ep["length"], "sec": ep["length"] / ds.fps,
                 "action_NaN": float(np.isnan(action).any(axis=1).mean()) if len(action) else 1.0,
                 "Y표시": index in rejected_episodes(root),
                 "task": ep["tasks"][0] if ep["tasks"] else ""})
summary = pd.DataFrame(rows)
print()
print(summary.to_string(index=False, formatters={"sec": "{:.1f}".format,
                                                 "action_NaN": "{:.0%}".format}))
print("\n총 %.1f 분" % (summary["sec"].sum() / 60))

## ③ 키보드 검수

아래 셀을 실행하면 `SG2 review` 창이 뜹니다. **창을 클릭해서 포커스를 준 뒤** 키를 누르세요
(주피터 셀에 포커스가 있으면 키가 전달되지 않습니다).

`K` 보관 · `D` 폐기 · `SPACE` 일시정지 · `R` 다시 · `N`/`P` 이동 · `Q` 종료

In [ ]:
# ③ 키보드 검수 — 창을 클릭한 뒤 키를 누르세요
WIN = "SG2 review"
TILE_H = 260          # 카메라 타일 높이(px). 화면이 좁으면 줄이세요
REVIEW = None         # 예: [0, 3, 5] 로 일부만 검수. None = 전체

decisions = globals().get("decisions", {})      # {episode_index: True(보관)/False(폐기)}
for marked in rejected_episodes(root):          # 녹화 중 Y 로 폐기 표시한 것
    decisions.setdefault(marked, False)
if rejected_episodes(root):
    print("녹화 중 Y 로 표시된 폐기 후보:", rejected_episodes(root))

def open_clip(episode_index):
    # 카메라별 VideoCapture 를 열고 이 에피소드의 시작 타임스탬프로 이동한다.
    ep = ds.meta.episodes[episode_index]
    streams = []
    for key in ds.meta.video_keys:
        cap = cv2.VideoCapture(str(root / ds.meta.get_video_file_path(episode_index, key)))
        cap.set(cv2.CAP_PROP_POS_MSEC, float(ep["videos/%s/from_timestamp" % key]) * 1000.0)
        streams.append((key, cap))
    return ep, streams

def read_tile(streams):
    # 카메라들을 같은 높이로 맞춰 가로로 붙인다. 하나라도 끝나면 None.
    tiles = []
    for key, cap in streams:
        ok, img = cap.read()
        if not ok:
            return None
        height, width = img.shape[:2]
        img = cv2.resize(img, (max(1, int(width * TILE_H / height)), TILE_H))
        cv2.putText(img, key, (6, 20), cv2.FONT_HERSHEY_SIMPLEX, .55, (60, 220, 60), 1, cv2.LINE_AA)
        tiles.append(img)
    return np.hstack(tiles) if tiles else None

def overlay(tile, episode_index, frame_index, total, paused, mark):
    bar = np.zeros((54, tile.shape[1], 3), np.uint8)
    state = {True: "KEEP", False: "DISCARD"}.get(mark, "-")
    cv2.putText(bar, "ep %d / %d   frame %d/%d   %s   %s"
                % (episode_index, ds.meta.total_episodes - 1, frame_index, total,
                   "PAUSED" if paused else "PLAY", state),
                (8, 20), cv2.FONT_HERSHEY_SIMPLEX, .55, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(bar, "K keep  D discard  SPACE pause  R restart  N/P move  Q quit",
                (8, 43), cv2.FONT_HERSHEY_SIMPLEX, .45, (170, 170, 170), 1, cv2.LINE_AA)
    return np.vstack([tile, bar])

order = list(range(ds.meta.total_episodes)) if REVIEW is None else list(REVIEW)
cv2.namedWindow(WIN, cv2.WINDOW_NORMAL)
position = 0
try:
    while 0 <= position < len(order):
        episode_index = order[position]
        ep, streams = open_clip(episode_index)
        total, frame_index, paused, verdict, tile = ep["length"], 0, False, None, None
        delay = max(1, int(1000 / ds.fps))
        while True:
            if not paused or tile is None:
                nxt = read_tile(streams)
                if nxt is None or frame_index >= total:      # 에피소드 끝 -> 멈추고 판정 대기
                    paused = True
                else:
                    tile, frame_index = nxt, frame_index + 1
            if tile is None:
                print("ep %d: 비디오를 읽을 수 없습니다" % episode_index)
                verdict = "skip"
                break
            cv2.imshow(WIN, overlay(tile, episode_index, frame_index, total, paused,
                                    decisions.get(episode_index)))
            key = cv2.waitKey(delay if not paused else 30) & 0xFF
            if key == ord(" "):
                paused = not paused
            elif key in (ord("k"), ord("K")):
                verdict = True; break
            elif key in (ord("d"), ord("D"), ord("x"), ord("X")):
                verdict = False; break
            elif key in (ord("r"), ord("R")):
                for _, cap in streams: cap.release()
                ep, streams = open_clip(episode_index)
                frame_index, paused, tile = 0, False, None
            elif key in (ord("n"), ord("N")):
                verdict = "skip"; break
            elif key in (ord("p"), ord("P")):
                verdict = "back"; break
            elif key in (ord("q"), ord("Q"), 27):
                verdict = "quit"; break
            elif key in (ord("."), 83) and paused:            # . 또는 오른쪽 화살표
                nxt = read_tile(streams)
                if nxt is not None and frame_index < total:
                    tile, frame_index = nxt, frame_index + 1
        for _, cap in streams:
            cap.release()
        if verdict == "quit":
            print("검수 종료 (ep %d 에서 중단)" % episode_index)
            break
        if verdict == "back":
            position = max(0, position - 1)
            continue
        if verdict in (True, False):
            decisions[episode_index] = verdict
            print("ep %d -> %s" % (episode_index, "KEEP" if verdict else "DISCARD"))
        position += 1
finally:
    cv2.destroyAllWindows(); cv2.waitKey(1)

print("\n판정된 에피소드 %d / %d" % (len(decisions), ds.meta.total_episodes))

## ④ 판정 확인

판정하지 않은 에피소드는 **보관**으로 처리됩니다. 잘못 누른 게 있으면
`decisions[3] = True` 처럼 직접 고치고 이 셀을 다시 실행하세요.

In [ ]:
# ④ 판정 확인
keep = [i for i in range(ds.meta.total_episodes) if decisions.get(i, True)]
drop = [i for i in range(ds.meta.total_episodes) if not decisions.get(i, True)]
kept_frames = int(summary[summary["episode"].isin(keep)]["frames"].sum())
print("보관 %d개: %s" % (len(keep), keep))
print("폐기 %d개: %s" % (len(drop), drop))
print("보관 프레임 %d (%.1f 분)" % (kept_frames, kept_frames / ds.fps / 60))

## ⑤ 최종 데이터셋 생성

`delete_episodes()` 로 폐기한 에피소드를 뺀 **새 데이터셋**을 만듭니다. 원본은 그대로 남고,
보관한 에피소드는 0부터 다시 번호가 붙습니다.

In [ ]:
# ⑤ 최종 데이터셋 생성 (원본은 수정하지 않음)
from lerobot.datasets.dataset_tools import delete_episodes
import shutil

FINAL_NAME = "%s_final" % TASK_NAME
final_root = SAVE_ROOT / FINAL_NAME
final_repo = "local/%s" % FINAL_NAME

if not keep:
    raise SystemExit("보관할 에피소드가 없습니다.")
if final_root.exists():
    raise SystemExit("%s 가 이미 있습니다. 지우거나 FINAL_NAME 을 바꾸세요." % final_root)

if drop:
    delete_episodes(ds, episode_indices=drop, output_dir=final_root, repo_id=final_repo)
else:
    print("폐기할 에피소드가 없어 원본을 그대로 복사합니다.")
    shutil.copytree(root, final_root)
print("생성 완료:", final_root)

## ⑥ 최종 데이터셋 검증

새 데이터셋을 다시 열어 에피소드 수·프레임 수·샘플 shape 을 확인합니다.
학습 코드에서 열 때도 `video_backend="pyav"` 를 넘기세요 (이 PC 의 torchcodec 빌드가 깨져 있습니다).

In [ ]:
# ⑥ 최종 데이터셋 검증
final_ds = LeRobotDataset(final_repo, root=final_root, video_backend="pyav")
print("fps %d | 에피소드 %d | 프레임 %d" % (final_ds.fps, final_ds.meta.total_episodes, len(final_ds)))
for e in list_episodes(final_root):
    print("  episode_%06d: %4d frames  %5.1fs  task=%s"
          % (e["episode_index"], e["length"], e["length"] / final_ds.fps, e["tasks"]))

sample = final_ds[0]
print()
print("action", tuple(sample["action"].shape), "| observation.state", tuple(sample["observation.state"].shape))
for key in final_ds.meta.video_keys:
    print("%-36s %s" % (key, tuple(sample[key].shape)))
print()
print("학습 시:  LeRobotDataset(%r, root=%r, video_backend='pyav')" % (final_repo, str(final_root)))